# BanglaSQL — Colab Training Notebook
**Natural Language (Bangla) to SQL — University Management System**

### Steps
Run each cell **in order**. Runtime → Change runtime type → **T4 GPU** before starting.

## Step 1 — Check GPU

In [1]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU')

GPU available: True
GPU name: Tesla T4
VRAM: 15.6 GB


## Step 2 — Clone repository

In [3]:
# Replace with your actual GitHub repo URL
REPO_URL = 'https://github.com/mustafiz-07/BanglaSQL.git'

!git clone {REPO_URL} banglasql
%cd banglasql/
!ls -la

Cloning into 'banglasql'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 55 (delta 27), reused 45 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 55.91 KiB | 11.18 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/banglasql
total 128
drwxr-xr-x 4 root root  4096 Aug 28 05:31 .
drwxr-xr-x 1 root root  4096 Aug 28 05:31 ..
-rw-r--r-- 1 root root 25381 Aug 28 05:31 build_dataset.py
-rw-r--r-- 1 root root  7789 Aug 28 05:31 colab_train.ipynb
-rw-r--r-- 1 root root 14853 Aug 28 05:31 create_database.py
drwxr-xr-x 2 root root  4096 Aug 28 05:31 data
-rw-r--r-- 1 root root   232 Aug 28 05:31 docker-compose.yml
-rw-r--r-- 1 root root   360 Aug 28 05:31 Dockerfile
-rw-r--r-- 1 root root  2331 Aug 28 05:31 er_diagram.md
drwxr-xr-x 8 root root  4096 Aug 28 05:31 .git
-rw-r--r-- 1 root root   572 Aug 28 05:31 .gitignore
-rw-r--r-- 1 root root 11360 Aug 28 05:31 prepr

## Step 3 — Install dependencies

In [4]:
# NOTE: train.py's own Quick Start instructions call for
# requirements_colab.txt (a Colab-specific pin list — Colab already ships a
# working torch/CUDA build, so this avoids reinstalling a conflicting one).
# This cell previously referenced a plain 'requirements.txt', which doesn't
# match and would fail with 'file not found' on a fresh clone.
!pip install -r requirements_colab.txt -q
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 35.8 MB/s eta 0:00:00
Dependencies installed.


## Step 4 — Generate database & dataset

In [5]:
!python create_database.py

Creating schema in: /content/banglasql/banglasql.db
Populating with synthetic data...

BanglaSQL Database — Summary
  departments     :    10 rows
  instructors     :    50 rows
  students        :   304 rows
  courses         :    80 rows
  enrollments     :   471 rows
  attendance      :   500 rows

-- Sample: Top 5 students by CGPA --
  Tasnim Sultana — CGPA: 4.0
  Farhana Rahman — CGPA: 3.97
  Nazmul Alam — CGPA: 3.96
  Joynal Mondal — CGPA: 3.96
  Tasnim Begum — CGPA: 3.96

-- Sample: Students per department --
  Mechanical Engineering                        : 34 students
  Physics                                       : 34 students
  Computer Science and Engineering              : 33 students
  Business Administration                       : 32 students
  Civil Engineering                             : 32 students
  Mathematics                                   : 30 students
  Electrical and Electronic Engineering         : 29 students
  English                                   

In [6]:
!python build_dataset.py

BanglaSQL Dataset Builder — Phase 2

[1/5] Loaded 165 base templates
      Easy: 65
      Medium: 100

[2/5] After augmentation: 2475 pairs (before deduplication)

[3/5] After deduplication: 2475 pairs
      Easy:   975
      Medium: 1500

  [OK] Dataset size (2475 pairs) is within target range.

[4/5] Split:
      Train : 1381 (56%)
      Dev   : 577   (23%)
      Test  : 517  (21%)
      (Test patterns held out entirely from train for generalization)

[5/5] Saved /content/banglasql/data/dataset_train.json

[5/5] Saved /content/banglasql/data/dataset_dev.json

[5/5] Saved /content/banglasql/data/dataset_test.json

      Saved stats: /content/banglasql/data/dataset_stats.json

Dataset Stats Summary
{
  "base_templates": {
    "easy": 65,
    "medium": 100,
    "total": 165
  },
  "after_augmentation_dedup": {
    "easy": 975,
    "medium": 1500,
    "total": 2475
  },
  "splits": {
    "train": 1381,
    "dev": 577,
    "test": 517
  },
  "train_easy": 451,
  "train_medium": 930,
  "de

## Step 5 — Tokenizer analysis & preprocessing

In [7]:
!python preprocess_check.py

BanglaSQL — Phase 3: Preprocessing & Tokenizer Analysis

[1/3] Unicode Normalization (NFC)...
  train: 1381 pairs, 0 normalized
  dev: 577 pairs, 0 normalized
  test: 517 pairs, 0 normalized

  (Analysis below uses train+dev only — 1958 pairs. Test split (517 pairs) is excluded from here on so hyperparameter choices can't leak information from it.)

[2/3] Tokenizer Analysis...

  Loading tokenizer: csebuetnlp/banglat5
config.json: 100% 659/659 [00:00<00:00, 4.00MB/s]
tokenizer_config.json: 100% 1.83k/1.83k [00:00<00:00, 6.17MB/s]

spiece.model: downloading bytes:   0% 0.00/1.11M [00:00<?, ?B/s]
spiece.model: downloading bytes: 100% 697k/697k [00:00<00:00, 925kB/s, 68.7kB/s  ]
spiece.model: reconstructing file: 100% 1.11M/1.11M [00:00<00:00, 1.48MB/s,  110kB/s  ]
special_tokens_map.json: 100% 1.79k/1.79k [00:00<00:00, 7.74MB/s]

  Model: csebuetnlp/banglat5
  Vocab size: 32,100
  Bangla-script tokens in vocab: 28,644 (89.2%)
  Avg tokens per question (sample of 50): 13.8
  Avg fragment 

## Step 6 — Train
> Expected time: ~45–70 min on a T4 for a 40-epoch run over ~640 training
> pairs (exact split sizes are printed by build_dataset.py).
> Early stopping monitors eval_exact_match with patience=6 and an 8-epoch
> warm-up; the best checkpoint is restored and saved to checkpoints/best_model.
> Dev evaluation uses beam search (num_beams=4), matching inference.

In [ ]:
!python train.py

2026-08-28 05:33:13,341 [INFO] Loaded train config from /content/banglasql/data/train_config.json
2026-08-28 05:33:13,341 [INFO] ============================================================
2026-08-28 05:33:13,341 [INFO] BanglaSQL — Phase 3: Training
2026-08-28 05:33:13,341 [INFO] ============================================================
2026-08-28 05:33:13,341 [INFO] Model       : csebuetnlp/banglat5
2026-08-28 05:33:13,341 [INFO] Max input   : 256
2026-08-28 05:33:13,341 [INFO] Max target  : 128
2026-08-28 05:33:13,341 [INFO] Batch size  : 8 (x2 grad accum = 16 effective)
2026-08-28 05:33:13,341 [INFO] Epochs      : 15
2026-08-28 05:33:13,341 [INFO] LR          : 5e-05
2026-08-28 05:33:13,341 [INFO] Early stop  : patience=4, warm-up=5 epochs
2026-08-28 05:33:13,341 [INFO] Device      : cuda
2026-08-28 05:33:13,364 [INFO] GPU Name    : Tesla T4
2026-08-28 05:33:13,364 [INFO] 
Loading tokenizer & model: csebuetnlp/banglat5
2026-08-28 05:33:13,521 [INFO] HTTP Request: HEAD https://hu

## Step 7 — Save model to Google Drive (prevents loss on session timeout)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

DRIVE_SAVE_DIR = '/content/drive/MyDrive/BanglaSQL/checkpoints3'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Copy the best model checkpoint
src = 'checkpoints/best_model'
dst = os.path.join(DRIVE_SAVE_DIR, 'best_model')

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Best model saved to Google Drive: {dst}')
else:
    print('No best_model found. Check that training completed successfully.')

## Step 8 — Quick inference test (verify the model works)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from common import format_input, load_config

cfg       = load_config()
MODEL_DIR = 'checkpoints/best_model'

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR)
model.eval()

def predict(bangla_question: str) -> str:
    inp = format_input(bangla_question, cfg['schema_string'])
    ids = tokenizer(inp, return_tensors='pt',
                    max_length=cfg['max_input_length'], truncation=True)
    out = model.generate(**ids, max_length=cfg['max_target_length'],
                         num_beams=4, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

test_questions = [
    'সকল শিক্ষার্থীর তালিকা দাও।',
    'যেসব শিক্ষার্থীর CGPA ৩.৫-এর বেশি তাদের নাম দাও।',
    'প্রতিটি বিভাগে কতজন শিক্ষার্থী আছে তা দেখাও।',
]

print('=== Inference Test ===\n')
for q in test_questions:
    print(f'Q  : {q}')
    print(f'SQL: {predict(q)}\n')

## Step 9 — Evaluate on the test split (Phase 4)
Runs every generated query against the SQLite database and reports
execution accuracy, exact match, validity rate, per-component accuracy
and a failure-category breakdown. Results land in `logs/test_results.json`
and per-example predictions in `logs/test_predictions.json` — these are
the tables and the error-analysis material for the report.

In [ ]:
!python evaluate.py --split test

In [ ]:
import json

with open('logs/test_results.json') as f:
    results = json.load(f)

m = results['metrics']
print(f"Execution accuracy : {m['execution_accuracy']:.1%}")
print(f"Exact match        : {m['exact_match']:.1%}")
print(f"Validity rate      : {m['validity_rate']:.1%}")

print('\nBy difficulty:')
for tier, s in results['by_difficulty'].items():
    print(f"  {tier:8s} n={s['n']:4d}  exec={s['execution_accuracy']:.1%}")

print('\nFailure categories:')
for name, count in results['failure_categories'].items():
    print(f'  {name:22s} {count}')

### Training curves (for the report)
Plots loss and dev exact-match per epoch from `logs/train_history.json`.

In [ ]:
import json
import matplotlib.pyplot as plt

with open('logs/train_history.json') as f:
    history = json.load(f)

train_pts = [(h['epoch'], h['loss']) for h in history if 'loss' in h]
eval_pts  = [(h['epoch'], h['eval_loss']) for h in history if 'eval_loss' in h]
em_pts    = [(h['epoch'], h['eval_exact_match']) for h in history if 'eval_exact_match' in h]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(*zip(*train_pts), label='train loss')
ax1.plot(*zip(*eval_pts), label='dev loss')
ax1.set_xlabel('epoch'); ax1.set_ylabel('loss'); ax1.legend(); ax1.set_title('Loss')
ax2.plot(*zip(*em_pts), color='green')
ax2.set_xlabel('epoch'); ax2.set_ylabel('exact match'); ax2.set_title('Dev exact match')
plt.tight_layout()
plt.savefig('logs/training_curves.png', dpi=150)
plt.show()

## Step 10 — Download model
If you prefer downloading the checkpoint directly instead of using Drive:

In [ ]:
import shutil
shutil.make_archive('best_model', 'zip', 'checkpoints/best_model')

from google.colab import files
files.download('best_model.zip')